# TEAM NEXUS · Submission (Hackathon IA PUCP 2026)
Downscaling GFS (~27 km, 17×13) → WRF (3 km, 70×35) de RH, T, U, V en la Cordillera Blanca.

**Qué hace este notebook (de punta a punta, solo con la data oficial):**
1. Escribe `modelo.py` (la misma red que usa el producto en `app/`).
2. Entrena el ensamble final con **todo** `train.npz` (cada configuración, 5 semillas, hasta su mejor época
   elegida con validación temporal 16–30 jun).
3. Predice el test y escribe `submission.csv`; guarda los pesos y `stats.npz` en `models/` (los que carga la app).

**Cómo correrlo:** en Kaggle (GPU T4, agregar la competencia `hackathon-ia-pucp-2026` como input) o en local
con la data en `data/` (rutas relativas). Con `VALIDAR = True` además reentrena con corte temporal y muestra el
score en validación (duplica el tiempo).

**Idea del modelo:** CNN sobre 3 horas del GFS (t−2, t−1, t) + hora del día → interpolación a la grilla de 3 km →
se suman relieve (DEM) y un mapa aprendido → la red predice la corrección sobre una **base física**:
climatología WRF por hora + anomalía del GFS interpolada (*ancla GFS*), para que el cambio de época pase directo.

## 1. Configuración

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
CONFIGS = {"Gg": {"w": 160, "enc": 2, "wd": 0.001, "anchor": "gfs"}, "Igr": {"w": 128, "enc": 1, "wd": 0.0001, "anchor": "gfs", "rep": 3}, "Hg": {"w": 192, "enc": 2, "wd": 0.001, "anchor": "gfs"}, "I": {"w": 128, "enc": 1, "wd": 0.0001}, "Ggr": {"w": 160, "enc": 2, "wd": 0.001, "anchor": "gfs", "rep": 3}}  # w = ancho, enc = bloques del codificador, wd = weight decay, anchor, rep = junio ×rep
EPOCHS = {"Gg": 15, "Igr": 14, "Hg": 12, "I": 21, "Ggr": 12}  # mejor época de cada config en validación temporal (pipeline.py)
LR, BS, EPOCHS_PLAN = 2e-3, 32, 25  # OneCycle planificado a 25 épocas y cortado en EPOCHS[c] (igual que pipeline.py)
VALIDAR = False  # True: entrena también con corte temporal y reporta el score en val (val de referencia: 0.393)
OUT = '.'

## 2. Modelo (mismo archivo que usa el producto)

In [ ]:
%%writefile modelo.py
"""Modelo de downscaling GFS (17x13) -> WRF (70x35) de TEAM NEXUS.
Lo usan el entrenamiento (pipeline.py) y el producto (app/), así ambos dan exactamente lo mismo."""
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

VARS = ['RH', 'T', 'U', 'V']


def hour(t):
    return np.array([int(str(k)[-2:]) for k in t])


def windows(X, t):
    """(N,17,13,4) -> (N,3,17,13,4) con las horas t-2, t-1, t. Si falta una hora previa, repite la actual."""
    ts = pd.to_datetime(np.asarray(t).astype(str), format='%Y%m%d%H'); pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag)); out[i, j] = X[idx] if idx is not None else X[i]
    return out


def climatology(Y, t):
    h = hour(t); return np.stack([Y[h == k].mean(0) for k in range(24)])  # (24,70,35,4)


def make_stats(X, Y, t, static):
    """Todo lo que el modelo necesita además de los pesos. Se guarda en models/stats.npz."""
    clim = climatology(Y, t)
    s = dict(mu_x=X.mean((0, 1, 2)), sd_x=X.std((0, 1, 2)), clim=clim,
             anom_sd=(Y - clim[hour(t)]).std((0, 1, 2)), gclim=climatology(X, t),
             dem=static['dem'], gfs_lats=static['gfs_lats'], gfs_lons=static['gfs_lons'],
             wrf_lats=static['wrf_lats'], wrf_lons=static['wrf_lons'])
    s['anom_sd_gfs'] = (Y - base(s, X, t, 'gfs')).std((0, 1, 2))
    return s


def wrf_grid(stats):
    """Coordenadas de la grilla WRF en el sistema [-1,1] de grid_sample sobre la grilla GFS. (1,70,35,2)"""
    LAT, LON = np.meshgrid(stats['wrf_lats'], stats['wrf_lons'], indexing='ij')
    glat, glon = stats['gfs_lats'], stats['gfs_lons']  # GFS fila 0 = norte; WRF fila 0 = sur
    gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1; gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
    return torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32)


def base(stats, X, t, anchor='clim', bs=512):
    """Punto de partida que la red corrige. 'clim': climatología por hora (feb-jun).
    'gfs': climatología + anomalía del GFS interpolada a 3 km, así el cambio de época del GFS pasa directo.
    X: (N,17,13,4) físico (la hora t)."""
    h = hour(t); B = stats['clim'][h].astype(np.float32)
    if anchor == 'gfs':
        g = wrf_grid(stats)
        for s in range(0, len(X), bs):
            a = torch.tensor((X[s:s + bs] - stats['gclim'][h[s:s + bs]]).astype(np.float32)).permute(0, 3, 1, 2)
            B[s:s + bs] += F.grid_sample(a, g.expand(len(a), -1, -1, -1), align_corners=True).permute(0, 2, 3, 1).numpy()
    return B


def anom_sd(stats, anchor='clim'):
    return stats['anom_sd_gfs'] if anchor == 'gfs' else stats['anom_sd']


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.GELU(), nn.Conv2d(o, o, 3, padding=1), nn.GELU())


class Net(nn.Module):
    """CNN sobre el GFS -> interpolación a 3 km -> + relieve + mapa aprendido -> anomalía normalizada."""
    def __init__(self, stats, w=64, enc=1, anchor='clim', dec=1):
        super().__init__()
        self.anchor = anchor
        dem = stats['dem']
        self.register_buffer('grid', wrf_grid(stats), persistent=False)
        self.register_buffer('dem', torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32)[None, None], persistent=False)
        self.enc = nn.Sequential(block(14, w), *[block(w, w) for _ in range(enc - 1)])
        self.learned = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
                                 nn.Conv2d(w, w, 3, padding=1), nn.GELU(), *[block(w, w) for _ in range(dec - 1)], nn.Conv2d(w, 4, 1))  # dec=1: igual que antes (mismos pesos)

    def forward(self, x, h):  # x: (B,3,17,13,4) normalizado, h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = F.grid_sample(self.enc(torch.cat([x, hh], 1)), self.grid.expand(B, -1, -1, -1), align_corners=True)
        f = torch.cat([f, self.dem.expand(B, -1, -1, -1), self.learned.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)


@torch.no_grad()
def predict(models, stats, W, t, bs=256):
    """Promedio del ensamble en unidades físicas. models: lista de listas (una por configuración)."""
    dev = next(models[0][0].parameters()).device
    tt = lambda v: torch.tensor(v, dtype=torch.float32, device=dev)
    h_all = hour(t); P = 0
    for grupo in models:  # cada configuración puede tener otra base (ancla); se promedia en unidades físicas
        anc = grupo[0].anchor; B, ASD = base(stats, W[:, -1], t, anc), tt(anom_sd(stats, anc)); out = []
        for s in range(0, len(W), bs):
            x = tt((W[s:s + bs] - stats['mu_x']) / stats['sd_x']); h = torch.tensor(h_all[s:s + bs], device=dev)
            out.append((torch.stack([m.eval()(x, h.float()) for m in grupo]).mean(0) * ASD).cpu().numpy())
        P = P + (B + np.concatenate(out)) / len(models)
    P = np.array(P, dtype=np.float32)
    P[..., 0] = P[..., 0].clip(0, 100)  # la RH física (y la del WRF) está entre 0 y 100 %
    return P


def load_ensemble(folder, device='cpu'):
    """Carga models/stats.npz + models/config.json + pesos. Devuelve (models, stats)."""
    import json, os
    stats = dict(np.load(os.path.join(folder, 'stats.npz')))
    cfg = json.load(open(os.path.join(folder, 'config.json')))
    models = []
    for name, c in cfg['configs'].items():
        grupo = []
        for seed in cfg['seeds']:
            m = Net(stats, c['w'], c['enc'], c.get('anchor', 'clim'), c.get('dec', 1)).to(device)
            m.load_state_dict(torch.load(os.path.join(folder, f'{name}_s{seed}.pt'), map_location=device))
            grupo.append(m.eval())
        models.append(grupo)
    return models, stats


## 3. Data

In [ ]:
import glob, json, os, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from modelo import VARS, hour, windows, make_stats, Net, predict, base, anom_sd

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', dev, '· torch', torch.__version__)

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/{name}') + glob.glob(f'data/{name}')
    assert hits, f'No encuentro {name}: poner la data oficial en data/ o como input de Kaggle'
    return hits[0]

tr, te, st = np.load(find('train.npz')), np.load(find('test.npz')), dict(np.load(find('static.npz')))
Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)  # (N,3,17,13,4): horas t-2, t-1, t
print('train', Xtr.shape, Ytr.shape, ttr[0], '→', ttr[-1], '· test', Xte.shape, tte[0], '→', tte[-1])

## 4. Entrenamiento

In [ ]:
def train_one(cfg, seed, idx, stats, epochs):
    torch.manual_seed(seed); np.random.seed(seed)
    anc = cfg.get('anchor', 'clim'); m = Net(stats, cfg['w'], cfg['enc'], anc, cfg.get('dec', 1)).to(dev)
    idx = np.concatenate([idx] + [idx[ttr[idx] >= '2024060100']] * (cfg.get('rep', 1) - 1))  # junio repetido
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=cfg['wd'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS_PLAN * int(np.ceil(len(idx) / BS)))
    tt = lambda v: torch.tensor(v, dtype=torch.float32, device=dev)
    Btr, ASD = base(stats, Xtr, ttr, anc), tt(anom_sd(stats, anc))
    for ep in range(epochs):
        m.train(); np.random.shuffle(idx)
        for s in range(0, len(idx), BS):
            b = idx[s:s + BS]
            x = tt((Wtr[b] - stats['mu_x']) / stats['sd_x']); h = torch.tensor(hour(ttr[b]), device=dev).float()
            loss = F.mse_loss(m(x, h), tt(Ytr[b] - Btr[b]) / ASD)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return m.eval()

## 5. (Opcional) Validación temporal: entrena hasta el 15 jun y evalúa 16–30 jun

In [ ]:
if VALIDAR:
    val = ttr >= '2024061600'; trn = np.where(~val)[0]
    st_trn = make_stats(Xtr[~val], Ytr[~val], ttr[~val], st)
    grupos_v = [[train_one(CONFIGS[c], s, trn, st_trn, EPOCHS[c]) for s in SEEDS] for c in CONFIGS]
    P = predict(grupos_v, st_trn, Wtr[val], ttr[val])
    clim_rmse = np.sqrt(((st_trn['clim'][hour(ttr[val])] - Ytr[val]) ** 2).mean((0, 1, 2)))
    rmse = np.sqrt(((P - Ytr[val]) ** 2).mean((0, 1, 2)))
    print('RMSE', dict(zip(VARS, rmse.round(3))), '· score val', round(float(1 - (rmse / clim_rmse).mean()), 4))

## 6. Entrenamiento final con todo train + pesos para el producto

In [ ]:
os.makedirs(f'{OUT}/models', exist_ok=True)
st_all = make_stats(Xtr, Ytr, ttr, st)
np.savez(f'{OUT}/models/stats.npz', **st_all)  # mu_x, sd_x, climatologías, DEM y coordenadas: la app los necesita
grupos, t0 = [], time.time()
for c in CONFIGS:
    g = []
    for s in SEEDS:
        m = train_one(CONFIGS[c], s, np.arange(len(Xtr)), st_all, EPOCHS[c])
        torch.save(m.state_dict(), f'{OUT}/models/{c}_s{s}.pt'); g.append(m)
    grupos.append(g); print(f'{c}: {len(g)} semillas · {time.time() - t0:.0f} s', flush=True)
json.dump(dict(configs=CONFIGS, seeds=SEEDS, epochs=EPOCHS), open(f'{OUT}/models/config.json', 'w'), indent=1)

## 7. Submission

In [ ]:
P = predict(grupos, st_all, Wte, tte)  # (N_test,70,35,4) en unidades físicas, fila 0 = sur
sample = pd.read_csv(find('sample_submission.csv'))
parts = sample['id'].str.split('_', expand=True)  # id = YYYYMMDDHH_fila_columna
ti = parts[0].map({k: i for i, k in enumerate(tte)}).values
assert not pd.isna(ti).any(), 'hay horas del sample que no están en el test'
r, c = parts[1].astype(int).values, parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS): sub[v] = P[ti.astype(int), r, c, k].round(3)
assert len(sub) == len(sample) and not sub[VARS].isna().any().any()
assert 200 < sub['T'].min() and sub['T'].max() < 320, 'T debe estar en K'
assert sub['RH'].between(0, 100).all()
sub.to_csv(f'{OUT}/submission.csv', index=False)
print(sub.shape); sub.describe().round(2)